# The Training Loop

Full training loop in NumPy: batches, epochs, forward pass, backprop, weight updates.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Generate synthetic binary classification data

In [ ]:
np.random.seed(42)
n = 100
X = np.random.randn(n, 2)
y = (X[:, 0] + X[:, 1] > 0).astype(int)

def one_hot(y, n_classes=2):
    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1
    return Y

Y = one_hot(y)
print('X shape:', X.shape, '  Y shape:', Y.shape)


## 2. Initialize network and helpers

In [ ]:
W1 = np.random.randn(4, 2) * 0.1
b1 = np.zeros(4)
W2 = np.random.randn(2, 4) * 0.1
b2 = np.zeros(2)

def relu(z): return np.maximum(0, z)
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


## 3. Training loop with mini-batches

In [ ]:
lr = 0.05
epochs = 100
batch_size = 10
losses = []

for epoch in range(epochs):
    idx = np.random.permutation(n)
    epoch_loss = 0
    for i in range(0, n, batch_size):
        xb = X[idx[i:i+batch_size]]
        yb = Y[idx[i:i+batch_size]]
        # Forward
        z1 = xb @ W1.T + b1
        a1 = relu(z1)
        z2 = a1 @ W2.T + b2
        a2 = softmax(z2)
        loss = -np.mean(np.sum(yb * np.log(a2 + 1e-9), axis=1))
        epoch_loss += loss
        # Backward
        d2 = (a2 - yb) / len(xb)
        dW2 = d2.T @ a1; db2 = d2.sum(0)
        d1 = (d2 @ W2) * (z1 > 0)
        dW1 = d1.T @ xb; db1 = d1.sum(0)
        # Update
        W2 -= lr * dW2; b2 -= lr * db2
        W1 -= lr * dW1; b1 -= lr * db1
    losses.append(epoch_loss)

plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training loss curve')
plt.grid(alpha=0.3); plt.show()
print(f'Final loss: {losses[-1]:.4f}')


## 4. Track accuracy

In [ ]:
z1_all = X @ W1.T + b1
a1_all = relu(z1_all)
z2_all = a1_all @ W2.T + b2
a2_all = softmax(z2_all)
preds = a2_all.argmax(axis=1)
acc = np.mean(preds == y)
print(f'Training accuracy: {acc*100:.1f}%')


## 5. Add Adam optimizer
Replace SGD with Adam for faster convergence.

In [ ]:
np.random.seed(42)
W1_a = np.random.randn(4, 2) * 0.1; b1_a = np.zeros(4)
W2_a = np.random.randn(2, 4) * 0.1; b2_a = np.zeros(2)
params = [W1_a, b1_a, W2_a, b2_a]
m_list = [np.zeros_like(p) for p in params]
v_list = [np.zeros_like(p) for p in params]
lr_adam = 0.01; b1, b2_c = 0.9, 0.999; eps = 1e-8
adam_losses = []; t = 0

for epoch in range(100):
    idx = np.random.permutation(n)
    el = 0
    for i in range(0, n, 10):
        t += 1
        xb = X[idx[i:i+10]]; yb = Y[idx[i:i+10]]
        z1 = xb @ W1_a.T + b1_a; a1 = relu(z1)
        z2 = a1 @ W2_a.T + b2_a; a2 = softmax(z2)
        loss = -np.mean(np.sum(yb * np.log(a2+1e-9), axis=1)); el += loss
        d2 = (a2-yb)/len(xb); dW2=d2.T@a1; db2=d2.sum(0)
        d1=(d2@W2_a)*(z1>0); dW1=d1.T@xb; db1=d1.sum(0)
        grads = [dW1, db1, dW2, db2]
        for j, (p, g) in enumerate(zip(params, grads)):
            m_list[j] = b1*m_list[j]+(1-b1)*g
            v_list[j] = b2_c*v_list[j]+(1-b2_c)*g**2
            mh = m_list[j]/(1-b1**t); vh = v_list[j]/(1-b2_c**t)
            params[j] -= lr_adam*mh/(np.sqrt(vh)+eps)
        W1_a,b1_a,W2_a,b2_a = params
    adam_losses.append(el)

plt.plot(losses, label='SGD'); plt.plot(adam_losses, label='Adam')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('SGD vs Adam')
plt.legend(); plt.show()
